In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import xarray as xr
import rasterio
from rasterio.transform import rowcol
import numpy as np
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt


root = Path.cwd()

def get_canopy_species_pct_ba(trees,species_code,species_name):
    ## retain only dominiant and codominant trees
    keep = ['Plot','Tag_num','Species','DBH','Crown Class']
    j = trees['overstory_2023'][keep].merge(trees['overstory_2024'][keep],on=['Plot','Tag_num','Species']).merge(trees['overstory_2025'][keep],on=['Plot','Tag_num','Species'])
    j['dbh'] = np.round(j[['DBH_x','DBH_y','DBH']].mean(axis=1),2)
    j = j.replace({'s':'S','i':'I','c':'C','d':'D'})
    all_trees = j[['Plot','Tag_num','Species','dbh','Crown Class']]
    
    canopy = all_trees.loc[all_trees['Crown Class'].isin(['C','D'])].copy()
    # get basal area in square meters
    canopy["ba_m2"] = 0.00007854 * (canopy["dbh"]**2)

    # get percent beech basal area per plot
    total = canopy.groupby('Plot')['ba_m2'].sum().rename('total_ba')
    species = canopy.loc[canopy['Species']==species_code].groupby('Plot')['ba_m2'].sum().rename(f'{species_name}_ba')

    t = pd.concat([total,species],axis=1).reset_index() 

    t[f'{species_name}_ba'] = t[f'{species_name}_ba'].fillna(0) ## plots with no target species will be nan so fill with 0

    ## get percent beech per plot
    t[f'{species_name}_pct_ba'] = t[f'{species_name}_ba'] / t['total_ba']

    return t

def merge_health_status(status_df,dbh_df,year,canopy):
    keep = ['Plot', 'Tag_num', 'Species',
       'Crown Class', 'Leaf Density', 'Dieback Overall',
       'Leaf Discolor', 'Dead Main Cover Class', 'Dead Main Position',
       'Dead Fine Cover Class', 'Dead Fine Position', 'Normal Size No Symptom',
       'Normal Size No Symptom position', 'Normal Size striped',
       'normal size striped position', 'shrunken or curled',
       'shrunken or curled position']
    status = status_df[f'overstory_{year}'][keep].copy()
    status['year'] = year

    m = dbh_df.merge(status,on=['Plot','Tag_num','Species','Crown Class'])

    if canopy:
    
      m = m.loc[m['Crown Class'].isin(['C','D'])].copy()

    return m

def get_mean_dbh(trees):
    keep = ['Plot','Tag_num','Species','DBH','Crown Class']
    j = trees['overstory_2023'][keep].merge(trees['overstory_2024'][keep],on=['Plot','Tag_num','Species']).merge(trees['overstory_2025'][keep],on=['Plot','Tag_num','Species'])
    j['dbh'] = np.round(j[['DBH_x','DBH_y','DBH']].mean(axis=1),2)
    j = j.replace({'s':'S','i':'I','c':'C','d':'D'})
    all_trees = j[['Plot','Tag_num','Species','dbh','Crown Class']]

    return all_trees


In [ ]:
## add points and polygons to justin's plots
from shapely.geometry import Point

p = pd.read_excel(root.parent / 'data' / 'raw' / 'BLD_all_plot_locations.xlsx')
geom = [Point(lon,lat) for lon, lat in zip(p['Longitude'],p['Latitude'])]
geo_points = gpd.GeoDataFrame(p[['Site','Name']],geometry=geom,crs=4326  )
geo_points = geo_points.to_crs(26918)

# make polygons
geo_polys = geo_points.copy()
geo_polys['buffer'] = geo_polys.buffer(11.6)
geo_polys = geo_polys.drop(columns='geometry').rename(columns={'buffer':'geometry'}).set_geometry('geometry')

In [ ]:


trees = pd.read_excel(root / 'data' / 'raw'/ 'overstory_2023-2025.xlsx',sheet_name=None)




In [ ]:
def get_plot_level_metrics(trees,year):
    plot_col    = 'Plot'
    species_col = 'Species'
    species_code   = 'FAGR'
    ba_col      = 'ba_m2'                                  
    bld_metrics = ['no_symptom','banded','curled'] 
    dieback_col = 'Dieback Overall'  

    df = trees[f'overstory_{year}']
    df = df.loc[df['Crown Class'].isin(['C','D'])].copy()
    df = df.rename(columns={'Normal Size No Symptom': 'no_symptom','Normal Size striped': 'banded','shrunken or curled':'curled'})

    df[ba_col] = 0.00007854 * (df["DBH"]**2)

    is_beech = df['Species'] == species_code

    ## code nonbeech as 0 for bld metrics
    df = df.fillna({m:0.0 for m in bld_metrics})

    # weight metrics by basal area
    for m in bld_metrics:
        df[f'{m}_weighted'] = np.where(is_beech,df[m]*df[ba_col],0.0)
    df['nonbeech_dieback_weighted'] = np.where(~is_beech,df[dieback_col]*df[ba_col],0.0)
    df['beech_ba'] = np.where(is_beech,df[ba_col],0.0)
    df['nonbeech_ba'] = np.where(~is_beech,df[ba_col],0.0)

    ## sum metrics per plot
    grouped = df.groupby(plot_col)
    out = pd.DataFrame({'total_ba':grouped[ba_col].sum(),
                    'beech_ba' : grouped['beech_ba'].sum(),
                        'nonbeech_ba': grouped['nonbeech_ba'].sum(),
                        'nonbeech_dieback': grouped['nonbeech_dieback_weighted'].sum(),
                        'n_beech': grouped[species_col].apply(lambda x: (x==species_code).sum()),
                        'n_nonbeech': grouped[species_col].apply(lambda x: (x!=species_code).sum())})
    for m in bld_metrics:
        out[m] = grouped[f'{m}_weighted'].sum()

    # beech abundance
    out['beech_rel_ba'] = out['beech_ba'] / out['total_ba']

    # weighted summed per-tree metrics divided by total plot basal area
    for m in bld_metrics:
        out[f'{m}_burden'] = out[m] / out['total_ba'] 
        out[f'{m}_mean_severity'] = out[m] / out['beech_ba'] ## beech only severity metric

    out['nonbeech_dieback_burden'] = out['nonbeech_dieback'] / out['total_ba']

    out = out.reset_index()

    out['year'] = year

    return out

In [ ]:
## calculate for all years and concatenate
f = []
for y in [2023,2024,2025]:
    d = get_plot_level_metrics(trees=trees,year=y)
    f.append(d)

out_df = pd.concat(f)


In [ ]:
site_map = {
    'BRF':  'Black Rock Forest',
    'CVC':  'Catskills Visitor Center',
    'HP':   'Hillside Park',
    'MC':   'Marshlands Conservancy',
    'MP':   'Mohonk Preserve',
    'MTA':  'Mountaintop Arboretum',
    'NYBG': 'New York Botanical Garden',
    'RSP':  'Rockefeller State Park',
    'TR':   'Teatown Reservation',
    'VCP':  'Van Cortlandt Park',
}

out_df['Site'] = out_df['Plot'].str.split('-').str[2].map(site_map)

metrics = ['banded_burden', 'curled_burden','no_symptom'] 
#metrics = ['banded_mean_severity','curled_mean_severity','no_symptom_mean_severity'] 
labels  = ['Banded', 'Curled','No Symptom']
colors  = ['C0', 'C1','C2']
years   = [2023, 2024, 2025]
sites   = sorted(out_df['Site'].unique())

fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=False)
axes = axes.ravel()

for i, s in enumerate(sites):
    ax = axes[i]
    site = out_df[out_df['Site'] == s]
    for m, l, c in zip(metrics, labels, colors):
        # individual plots
        for _, g in site.groupby('Plot'):
            g = g.sort_values('year')
            ax.plot(g['year'], g[m], color=c, alpha=0.3, linewidth=0.9)
        # site median
        med = site.groupby('year')[m].median()
        ax.plot(med.index, med.values, color=c, marker='o', linewidth=2, label=l)
    ax.set_title(s, fontsize=10)
    ax.grid(False)

# hide any leftover panels if a site is missing
for j in range(len(sites), len(axes)):
    axes[j].set_visible(False)

for ax in axes:
    ax.set_xticks(years)
for ax in axes[-5:]:
    ax.set_xlabel('Year')
for ax in axes[::5]:
    ax.set_ylabel('Plot burden')

handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='upper center', bbox_to_anchor=(0.5, 0.98),
           ncol=len(labels), frameon=False,title='thin lines = plots, bold = site median')

metric_type = 'Severity' if 'severity' in metrics[0] else 'Burden'

fig.suptitle(f'BLD Symptom {metric_type} by Site', y=1.0)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
from bld_src import config

h = pd.read_csv(config.PLOT_HEALTH_METRICS)

In [ ]:
h.loc[(h['beech_rel_ba']>=.50)&(h['year']==2023)]  ## 15 plots have gt 40% beech basal area, 26 are gt 20%, 13 are gt 50%